# OEPolars - Getting Started

This notebook demonstrates the basic usage of OEPolars for working with molecular data in Polars DataFrames.

## Prerequisites

```bash
pip install oepolars
```

**Note:** You'll need an OpenEye Toolkits license. Free academic licenses are available at [OpenEye Scientific](https://www.eyesopen.com/academic-licensing).

In [1]:
import polars as pl
import oepolars as oep
from openeye import oechem

## Creating DataFrames with Molecules

There are several ways to create DataFrames with molecular data:

### Method 1: Convert SMILES strings to molecules using `.chem.as_molecule()`

This converts a column with molecule strings to a `molecule` column with `oechem.OEMol` objects. The default format is SMILES, but you can read any molecule format supported by the OpenEye Toolkits:

```python
DataFrame.chem.as_molecule(
    columns: str | Iterable[str],
    molecule_format: int | None = None,
) -> DataFrame
```

In [2]:
# Create sample data from SMILES strings
df = pl.DataFrame({
    "SMILES": ["CC(=O)Oc1ccccc1C(=O)O", "CC(C)Cc1ccc(cc1)C(C)C(=O)O", 
               "CC(=O)Nc1ccc(cc1)O", "Cn1cnc2c1c(=O)n(c(=O)n2C)C"],
    "Name": ["Aspirin", "Ibuprofen", "Acetaminophen", "Caffeine"],
    "MW": [180.16, 206.28, 151.16, 194.19]
})

# Convert the SMILES column to molecules using the .chem accessor
df = df.chem.as_molecule("SMILES")

print(f"Created DataFrame with {len(df)} molecules")
print(f"SMILES column dtype: {df.schema['SMILES']}")
df

Created DataFrame with 4 molecules
SMILES column dtype: MoleculeType('molecule', Object)


SMILES,Name,MW
ext[molecule],str,f64
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x10c53f240> >,"""Aspirin""",180.16
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x1067db000> >,"""Ibuprofen""",206.28
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x1061d4510> >,"""Acetaminophen""",151.16
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x1061d43f0> >,"""Caffeine""",194.19


### Method 2: Read from molecule files (SDF, SMILES, OEB)

If you use a native OEPolars reader, molecules will already be objects in the DataFrame.

```python
# Read from SDF file (includes SD data as columns)
df = oep.read_sdf("molecules.sdf", molecule_column="Mol", title_column="Name")

# Read from SMILES file
df = oep.read_smi("molecules.smi", molecule_column="Mol", title_column="Name")

# Read from OEB (OpenEye binary) file
df = oep.read_oeb("molecules.oeb", molecule_column="Mol", title_column="Name")
```

### Method 3: Read CSV with molecule strings

You can also read CSV files that have molecule columns. This will automatically create molecule objects.

```python
# Read CSV file and convert SMILES column to molecules
df = oep.read_molecule_csv("molecules.csv", smiles_column="SMILES")
```

## Working with Molecules

Once molecules are in a DataFrame, you can use standard Polars operations:

In [3]:
# Calculate the number of heavy atoms using map_elements
df = df.with_columns(
    pl.col("SMILES").map_elements(
        lambda mol: oechem.OECount(mol, oechem.OEIsHeavy()) if mol else 0,
        return_dtype=pl.Int64
    ).alias("NumHeavyAtoms")
)

# Molecules with 14+ heavy atoms
df.filter(pl.col("NumHeavyAtoms") >= 14)

SMILES,Name,MW,NumHeavyAtoms
ext[molecule],str,f64,i64
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x1067db000> >,"""Ibuprofen""",206.28,15
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x1061d43f0> >,"""Caffeine""",194.19,14


## Molecular Calculations

Examples of various calculations you can perform.

In [4]:
# Calculate molecular properties
df = df.with_columns([
    pl.col("SMILES").map_elements(
        lambda mol: mol.NumAtoms() if mol else None,
        return_dtype=pl.Int64
    ).alias("NumAtoms"),
    pl.col("SMILES").map_elements(
        lambda mol: mol.NumBonds() if mol else None,
        return_dtype=pl.Int64
    ).alias("NumBonds"),
    pl.col("SMILES").map_elements(
        lambda mol: oechem.OECount(mol, oechem.OEIsOxygen()) if mol else None,
        return_dtype=pl.Int64
    ).alias("NumOxygens")
])

df.select(["Name", "NumAtoms", "NumBonds", "NumOxygens"])

Name,NumAtoms,NumBonds,NumOxygens
str,i64,i64,i64
"""Aspirin""",13,13,4
"""Ibuprofen""",15,15,2
"""Acetaminophen""",11,11,2
"""Caffeine""",14,15,2


## Using the `.chem` Accessor

OEPolars provides convenient chemistry functions via the `.chem` accessor for Series, DataFrames, and LazyFrames.

### DataFrame Accessors

Access these methods via `df.chem.<method>()`:

| Method | Description |
|--------|-------------|
| `as_molecule(columns, molecule_format=None)` | Convert column(s) to MoleculeType |
| `filter_valid(columns)` | Filter rows to keep only those with valid molecules |
| `detect_molecule_columns(sample_size=25)` | Auto-detect and convert molecule columns |
| `write_sdf(filepath, molecule_column, ...)` | Write to SDF file |
| `write_smi(filepath, molecule_column, ...)` | Write to SMILES file |
| `write_oeb(filepath, molecule_column, ...)` | Write to OEB file |
| `write_oedb(filepath, molecule_column, ...)` | Write to OEDB file |
| `write_oedu(filepath, design_unit_column, ...)` | Write to OEDU file |
| `write_molecule_csv(filepath, molecule_column, ...)` | Write to CSV with SMILES |
| `write_parquet(filepath, molecule_columns=None, ...)` | Write to Parquet with serialization |

### Series Accessors

Access these methods via `series.chem.<method>()`:

| Method | Returns | Description |
|--------|---------|-------------|
| `is_valid()` | `Series[bool]` | Boolean mask of valid molecules |
| `to_smiles(flavor=...)` | `Series[str]` | Convert to SMILES strings |
| `substructure_search(pattern, adjustH=False)` | `Series[bool]` | Substructure search with SMARTS pattern |
| `copy_molecules()` | `Series[MoleculeType]` | Deep copy all molecules |
| `copy_design_units()` | `Series[DesignUnitType]` | Deep copy all design units |
| `get_ligands(clear_titles=False)` | `Series[MoleculeType]` | Extract ligand molecules from design units |
| `get_proteins(clear_titles=False)` | `Series[MoleculeType]` | Extract protein molecules from design units |
| `get_components(mask)` | `Series[MoleculeType]` | Extract components by mask from design units |

### LazyFrame Accessors

Access these methods via `lf.chem.<method>()`:

| Method | Description |
|--------|-------------|
| `has_molecule_columns()` | Check if LazyFrame has molecule columns |
| `molecule_columns()` | Get list of molecule column names |

**Note:** Most molecule operations require eager evaluation. Use `.collect()` first to convert LazyFrame to DataFrame.

In [5]:
# Generate canonical SMILES using the .chem accessor
canonical = df["SMILES"].chem.to_smiles()
print("Canonical SMILES:")
for name, smiles in zip(df["Name"].to_list(), canonical.to_list()):
    print(f"  {name}: {smiles}")

Canonical SMILES:
  Aspirin: CC(=O)Oc1ccccc1C(=O)O
  Ibuprofen: CC(C)Cc1ccc(cc1)C(C)C(=O)O
  Acetaminophen: CC(=O)Nc1ccc(cc1)O
  Caffeine: Cn1cnc2c1c(=O)n(c(=O)n2C)C


## Checking Molecule Validity

OEPolars provides methods to check and filter molecule validity:

In [6]:
# Create a DataFrame with some invalid molecules
df_test = pl.DataFrame({
    "SMILES": ["CCO", "invalid_smiles", "c1ccccc1"],
    "Name": ["Ethanol", "Invalid", "Benzene"]
})
df_test = df_test.chem.as_molecule("SMILES")

# Check which molecules are valid using .chem.is_valid()
validity = df_test["SMILES"].chem.is_valid()
print("Molecule validity:")
for name, is_valid in zip(df_test["Name"].to_list(), validity.to_list()):
    print(f"  {name}: {'Valid' if is_valid else 'Invalid'}")

Molecule validity:
  Ethanol: Valid
  Invalid: Invalid
  Benzene: Valid


In [7]:
# Filter to keep only valid molecules using .chem.filter_valid()
df_valid = df_test.chem.filter_valid("SMILES")
print(f"\nOriginal rows: {len(df_test)}")
print(f"After filtering: {len(df_valid)} valid molecules")
df_valid


Original rows: 3
After filtering: 2 valid molecules


SMILES,Name
ext[molecule],str
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x10e4588a0> >,"""Ethanol"""
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x10e43a6a0> >,"""Benzene"""


## Data Manipulation

Polars operations work seamlessly with molecular data:

In [8]:
# Sorting
sorted_df = df.sort("MW", descending=True)
print("Molecules sorted by molecular weight:")
sorted_df.select(["Name", "MW"])

Molecules sorted by molecular weight:


Name,MW
str,f64
"""Ibuprofen""",206.28
"""Caffeine""",194.19
"""Aspirin""",180.16
"""Acetaminophen""",151.16


In [9]:
# Grouping and aggregation
df = df.with_columns(
    pl.col("SMILES").map_elements(
        lambda mol: oechem.OECount(mol, oechem.OEAtomIsInRing()) > 0 if mol else False,
        return_dtype=pl.Boolean
    ).alias("HasRing")
)

ring_stats = df.group_by("HasRing").agg([
    pl.col("MW").mean().alias("MW_mean"),
    pl.col("MW").min().alias("MW_min"),
    pl.col("MW").max().alias("MW_max"),
    pl.col("NumAtoms").mean().alias("NumAtoms_mean")
])

print("Statistics by ring presence:")
ring_stats

Statistics by ring presence:


HasRing,MW_mean,MW_min,MW_max,NumAtoms_mean
bool,f64,f64,f64,f64
true,182.9475,151.16,206.28,13.25


## Copying Molecules

Create deep copies of molecular data when needed:

In [10]:
# Deep copy molecules using the .chem accessor
df = df.with_columns(
    df["SMILES"].chem.copy_molecules().alias("SMILES_copy")
)

print(f"Original and copied molecules are different objects: {df['SMILES'][0] is not df['SMILES_copy'][0]}")

Original and copied molecules are different objects: True


## Handling Missing Values

OEPolars handles null values gracefully:

In [11]:
# Add a row with missing molecule
df_with_null = pl.concat([
    df.select(["SMILES", "Name", "MW"]),
    pl.DataFrame({"SMILES": [None], "Name": ["Unknown"], "MW": [None]}, 
                 schema={"SMILES": df.schema["SMILES"], "Name": pl.String, "MW": pl.Float64})
])

print(f"DataFrame has {df_with_null['SMILES'].null_count()} missing molecules")

# Filter out missing values
df_clean = df_with_null.filter(pl.col("SMILES").is_not_null())
print(f"After filtering: {len(df_clean)} molecules remain")

DataFrame has 1 missing molecules
After filtering: 4 molecules remain


## Substructure Searching

Perform SMARTS-based substructure searches using `.chem.substructure_search()`:

In [12]:
# Find molecules containing a carboxylic acid group
carboxylic_acid_pattern = "C(=O)O"

matches = df["SMILES"].chem.substructure_search(carboxylic_acid_pattern)
print(f"Molecules containing carboxylic acid group:")
df.filter(matches).select(["Name", "MW"])

Molecules containing carboxylic acid group:


Name,MW
str,f64
"""Aspirin""",180.16
"""Ibuprofen""",206.28


## Converting Molecules to Different Formats

The `.chem` accessor provides methods to convert molecules to various string formats:

In [13]:
# Convert to SMILES strings
smiles_series = df["SMILES"].chem.to_smiles()
print("SMILES strings:")
print(smiles_series.head(2).to_list())

SMILES strings:
['CC(=O)Oc1ccccc1C(=O)O', 'CC(C)Cc1ccc(cc1)C(C)C(=O)O']


## Lazy Evaluation with LazyFrames

OEPolars supports Polars LazyFrames for query optimization. Use `scan_*` functions to create LazyFrames:

In [14]:
# LazyFrame example (would use scan_sdf, scan_oeb, etc. for files)
lf = df.lazy()

# Check for molecule columns
print(f"Has molecule columns: {lf.chem.has_molecule_columns()}")
print(f"Molecule columns: {lf.chem.molecule_columns()}")

# Collect to DataFrame for molecule operations
result = lf.filter(pl.col("MW") > 180).collect()
print(f"\nFiltered result: {len(result)} molecules")
result

Has molecule columns: True
Molecule columns: ['SMILES', 'SMILES_copy']

Filtered result: 3 molecules


SMILES,Name,MW,NumHeavyAtoms,NumAtoms,NumBonds,NumOxygens,HasRing,SMILES_copy
ext[molecule],str,f64,i64,i64,i64,i64,bool,ext[molecule]
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x10c53f240> >,"""Aspirin""",180.16,13,13,13,4,true,<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x10e458c00> >
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x1067db000> >,"""Ibuprofen""",206.28,15,15,15,2,true,<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x10e459980> >
<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x1061d43f0> >,"""Caffeine""",194.19,14,14,15,2,true,<oechem.OEMol; proxy of <Swig Object of type 'OEMolWrapper *' at 0x10e458ff0> >


## Parquet Serialization

OEPolars supports saving and loading molecules to/from Parquet files:

In [15]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmpdir:
    parquet_path = Path(tmpdir) / "molecules.parquet"
    
    # Write to Parquet (molecules are serialized to binary OEB format)
    df.select(["SMILES", "Name", "MW"]).chem.write_parquet(parquet_path)
    print(f"Wrote Parquet file: {parquet_path.stat().st_size} bytes")
    
    # Read back from Parquet (specify which columns contain molecules)
    df_read = oep.read_parquet(parquet_path, molecule_columns=["SMILES"])
    print(f"\nRead back {len(df_read)} molecules")
    print(f"SMILES column dtype: {df_read.schema['SMILES']}")
    df_read

Wrote Parquet file: 3422 bytes

Read back 4 molecules
SMILES column dtype: MoleculeType('molecule', Object)
